#  Base de Datos Jardinería → Staging


**Pasos:**
1. Crear y poblar la base de datos `jardineria`
2. Crear la base de datos `staging`
3. Migrar datos de Jardinería → Staging
4. Validaciones y conteos
5. Exportar scripts SQL y BK.

## CELDA 1 — Importar librerías y configurar utilidades

In [ ]:
import sqlite3
import os
import shutil
import pandas as pd
from google.colab import files

# Rutas de los archivos de base de datos
DB_JARDINERIA = '/content/jardineria.db'
DB_STAGING    = '/content/staging.db'

# Función auxiliar: ejecutar SQL y mostrar resultado
def run_query(conn, sql, titulo=None):
    try:
        df = pd.read_sql_query(sql, conn)
        if titulo:
            print(f"\n{'='*50}\n{titulo}\n{'='*50}")
        display(df)
        return df
    except Exception as e:
        print(f"Error: {e}")

def ejecutar_ddl(conn, sql, mensaje=None):
    try:
        conn.executescript(sql)
        conn.commit()
        if mensaje:
            print(f"✅ {mensaje}")
    except Exception as e:
        print(f" Error en DDL: {e}")

print(" Librerías cargadas correctamente")

 Librerías cargadas correctamente


## CELDA 2 — Crear base de datos Jardinería y sus tablas

In [ ]:
# Eliminar DB si ya existe para empezar limpio
if os.path.exists(DB_JARDINERIA):
    os.remove(DB_JARDINERIA)
    print("  jardineria.db eliminada (reinicio limpio)")

conn_j = sqlite3.connect(DB_JARDINERIA)
conn_j.execute("PRAGMA foreign_keys = ON")

ddl_jardineria = """
CREATE TABLE oficina (
  ID_oficina       INTEGER PRIMARY KEY AUTOINCREMENT,
  Descripcion      VARCHAR(10) NOT NULL,
  ciudad           VARCHAR(30) NOT NULL,
  pais             VARCHAR(50) NOT NULL,
  region           VARCHAR(50),
  codigo_postal    VARCHAR(10) NOT NULL,
  telefono         VARCHAR(20) NOT NULL,
  linea_direccion1 VARCHAR(50) NOT NULL,
  linea_direccion2 VARCHAR(50)
);

CREATE TABLE empleado (
  ID_empleado INTEGER PRIMARY KEY AUTOINCREMENT,
  nombre      VARCHAR(50) NOT NULL,
  apellido1   VARCHAR(50) NOT NULL,
  apellido2   VARCHAR(50),
  extension   VARCHAR(10) NOT NULL,
  email       VARCHAR(100) NOT NULL,
  ID_oficina  INTEGER NOT NULL REFERENCES oficina(ID_oficina),
  ID_jefe     INTEGER REFERENCES empleado(ID_empleado),
  puesto      VARCHAR(50)
);

CREATE TABLE Categoria_producto (
  Id_Categoria     INTEGER PRIMARY KEY AUTOINCREMENT,
  Desc_Categoria   VARCHAR(50) NOT NULL,
  descripcion_texto TEXT,
  descripcion_html  TEXT,
  imagen           VARCHAR(256)
);

CREATE TABLE cliente (
  ID_cliente              INTEGER PRIMARY KEY AUTOINCREMENT,
  nombre_cliente          VARCHAR(50) NOT NULL,
  nombre_contacto         VARCHAR(30),
  apellido_contacto       VARCHAR(30),
  telefono                VARCHAR(15) NOT NULL,
  fax                     VARCHAR(15) NOT NULL,
  linea_direccion1        VARCHAR(50) NOT NULL,
  linea_direccion2        VARCHAR(50),
  ciudad                  VARCHAR(50) NOT NULL,
  region                  VARCHAR(50),
  pais                    VARCHAR(50),
  codigo_postal           VARCHAR(10),
  ID_empleado_rep_ventas  INTEGER REFERENCES empleado(ID_empleado),
  limite_credito          NUMERIC(15,2)
);

CREATE TABLE pedido (
  ID_pedido      INTEGER PRIMARY KEY AUTOINCREMENT,
  fecha_pedido   DATE NOT NULL,
  fecha_esperada DATE NOT NULL,
  fecha_entrega  DATE,
  estado         VARCHAR(15) NOT NULL,
  comentarios    TEXT,
  ID_cliente     INTEGER NOT NULL REFERENCES cliente(ID_cliente)
);

CREATE TABLE producto (
  ID_producto      INTEGER PRIMARY KEY AUTOINCREMENT,
  CodigoProducto   VARCHAR(15) NOT NULL,
  nombre           VARCHAR(70) NOT NULL,
  Categoria        INTEGER NOT NULL REFERENCES Categoria_producto(Id_Categoria),
  dimensiones      VARCHAR(25),
  proveedor        VARCHAR(50),
  descripcion      TEXT,
  cantidad_en_stock SMALLINT NOT NULL,
  precio_venta     NUMERIC(15,2) NOT NULL,
  precio_proveedor NUMERIC(15,2)
);

CREATE TABLE detalle_pedido (
  ID_detalle_pedido INTEGER PRIMARY KEY AUTOINCREMENT,
  ID_pedido         INTEGER NOT NULL REFERENCES pedido(ID_pedido),
  ID_producto       INTEGER NOT NULL REFERENCES producto(ID_producto),
  cantidad          INTEGER NOT NULL,
  precio_unidad     NUMERIC(15,2) NOT NULL,
  numero_linea      SMALLINT NOT NULL
);

CREATE TABLE pago (
  ID_pago        INTEGER PRIMARY KEY AUTOINCREMENT,
  ID_cliente     INTEGER NOT NULL REFERENCES cliente(ID_cliente),
  forma_pago     VARCHAR(40) NOT NULL,
  id_transaccion VARCHAR(50) NOT NULL,
  fecha_pago     DATE NOT NULL,
  total          NUMERIC(15,2) NOT NULL
);
"""

ejecutar_ddl(conn_j, ddl_jardineria, "Tablas de Jardinería creadas")

  jardineria.db eliminada (reinicio limpio)
✅ Tablas de Jardinería creadas


## CELDA 3 — Poblar Jardinería con todos los datos

In [ ]:
cur = conn_j.cursor()

# ─── OFICINAS ────────────────────────────────────────────────
oficinas = [
    ('BCN-ES','Barcelona','España','Barcelona','08019','+34 93 3561182','Avenida Diagonal, 38','3A escalera Derecha'),
    ('BOS-USA','Boston','EEUU','MA','02108','+1 215 837 0825','1550 Court Place','Suite 102'),
    ('LON-UK','Londres','Inglaterra','EMEA','EC2N 1HN','+44 20 78772041','52 Old Broad Street','Ground Floor'),
    ('MAD-ES','Madrid','España','Madrid','28032','+34 91 7514487','Bulevar Indalecio Prieto, 32',''),
    ('PAR-FR','Paris','Francia','EMEA','75017','+33 14 723 4404',"29 Rue Jouffroy d'abbans",''),
    ('SFC-USA','San Francisco','EEUU','CA','94080','+1 650 219 4782','100 Market Street','Suite 300'),
    ('SYD-AU','Sydney','Australia','APAC','NSW 2010','+61 2 9264 2451','5-11 Wentworth Avenue','Floor #2'),
    ('TAL-ES','Talavera de la Reina','España','Castilla-LaMancha','45632','+34 925 867231','Francisco Aguirre, 32','5º piso (exterior)'),
    ('TOK-JP','Tokyo','Japón','Chiyoda-Ku','102-8578','+81 33 224 5000','4-1 Kioicho',''),
]
cur.executemany("INSERT INTO oficina(Descripcion,ciudad,pais,region,codigo_postal,telefono,linea_direccion1,linea_direccion2) VALUES (?,?,?,?,?,?,?,?)", oficinas)

# ─── EMPLEADOS ───────────────────────────────────────────────
empleados = [
    ('Marcos','Magaña','Perez','3897','marcos@jardineria.es',8,None,'Director General'),
    ('Ruben','López','Martinez','2899','rlopez@jardineria.es',8,1,'Subdirector Marketing'),
    ('Alberto','Soria','Carrasco','2837','asoria@jardineria.es',8,2,'Subdirector Ventas'),
    ('Maria','Solís','Jerez','2847','msolis@jardineria.es',8,2,'Secretaria'),
    ('Felipe','Rosas','Marquez','2844','frosas@jardineria.es',8,3,'Representante Ventas'),
    ('Juan Carlos','Ortiz','Serrano','2845','cortiz@jardineria.es',8,3,'Representante Ventas'),
    ('Carlos','Soria','Jimenez','2444','csoria@jardineria.es',4,3,'Director Oficina'),
    ('Mariano','López','Murcia','2442','mlopez@jardineria.es',4,7,'Representante Ventas'),
    ('Lucio','Campoamor','Martín','2442','lcampoamor@jardineria.es',4,7,'Representante Ventas'),
    ('Hilario','Rodriguez','Huertas','2444','hrodriguez@jardineria.es',4,7,'Representante Ventas'),
    ('Emmanuel','Magaña','Perez','2518','manu@jardineria.es',1,3,'Director Oficina'),
    ('José Manuel','Martinez','De la Osa','2519','jmmart@hotmail.es',1,11,'Representante Ventas'),
    ('David','Palma','Aceituno','2519','dpalma@jardineria.es',1,11,'Representante Ventas'),
    ('Oscar','Palma','Aceituno','2519','opalma@jardineria.es',1,11,'Representante Ventas'),
    ('Francois','Fignon','','9981','ffignon@gardening.com',5,3,'Director Oficina'),
    ('Lionel','Narvaez','','9982','lnarvaez@gardening.com',5,15,'Representante Ventas'),
    ('Laurent','Serra','','9982','lserra@gardening.com',5,15,'Representante Ventas'),
    ('Michael','Bolton','','7454','mbolton@gardening.com',6,3,'Director Oficina'),
    ('Walter Santiago','Sanchez','Lopez','7454','wssanchez@gardening.com',6,18,'Representante Ventas'),
    ('Hilary','Washington','','7565','hwashington@gardening.com',2,3,'Director Oficina'),
    ('Marcus','Paxton','','7565','mpaxton@gardening.com',2,20,'Representante Ventas'),
    ('Lorena','Paxton','','7665','lpaxton@gardening.com',2,20,'Representante Ventas'),
    ('Nei','Nishikori','','8734','nnishikori@gardening.com',9,3,'Director Oficina'),
    ('Narumi','Riko','','8734','nriko@gardening.com',9,23,'Representante Ventas'),
    ('Takuma','Nomura','','8735','tnomura@gardening.com',9,23,'Representante Ventas'),
    ('Amy','Johnson','','3321','ajohnson@gardening.com',3,3,'Director Oficina'),
    ('Larry','Westfalls','','3322','lwestfalls@gardening.com',3,26,'Representante Ventas'),
    ('John','Walton','','3322','jwalton@gardening.com',3,26,'Representante Ventas'),
    ('Kevin','Fallmer','','3210','kfalmer@gardening.com',7,3,'Director Oficina'),
    ('Julian','Bellinelli','','3211','jbellinelli@gardening.com',7,29,'Representante Ventas'),
    ('Mariko','Kishi','','3211','mkishi@gardening.com',7,29,'Representante Ventas'),
]
cur.executemany("INSERT INTO empleado(nombre,apellido1,apellido2,extension,email,ID_oficina,ID_jefe,puesto) VALUES (?,?,?,?,?,?,?,?)", empleados)

# ─── CATEGORÍAS ──────────────────────────────────────────────
categorias = [
    ('Herbaceas','Plantas para jardin decorativas',None,None),
    ('Herramientas','2 para todo tipo de acción',None,None),
    ('Aromaticas','Plantas aromáticas',None,None),
    ('Frutales','Árboles pequeños de producción frutal',None,None),
    ('Ornamentales','Plantas vistosas para la decoración del jardín',None,None),
]
cur.executemany("INSERT INTO Categoria_producto(Desc_Categoria,descripcion_texto,descripcion_html,imagen) VALUES (?,?,?,?)", categorias)

# ─── CLIENTES ────────────────────────────────────────────────
clientes = [
    ('GoldFish Garden','Daniel G','GoldFish','5556901745','5556901746','False Street 52 2 A',None,'San Francisco',None,'USA','24006',19,3000),
    ('Gardening Associates','Anne','Wright','5557410345','5557410346','Wall-e Avenue',None,'Miami','Miami','USA','24010',19,6000),
    ('Gerudo Valley','Link','Flaute','5552323129','5552323128','Oaks Avenue nº22',None,'New York',None,'USA','85495',22,12000),
    ('Tendo Garden','Akane','Tendo','55591233210','55591233211','Null Street nº69',None,'Miami',None,'USA','696969',22,600000),
    ('Lasas S.A.','Antonio','Lasas','34916540145','34914851312','C/Leganes 15',None,'Fuenlabrada','Madrid','Spain','28945',8,154310),
    ('Beragua','Jose','Bermejo','654987321','916549872','C/pintor segundo','Getafe','Madrid','Madrid','Spain','28942',11,20000),
    ('Club Golf Puerta del hierro','Paco','Lopez','62456810','919535678','C/sinesio delgado','Madrid','Madrid','Madrid','Spain','28930',11,40000),
    ('Naturagua','Guillermo','Rengifo','689234750','916428956','C/majadahonda','Boadilla','Madrid','Madrid','Spain','28947',11,32000),
    ('DaraDistribuciones','David','Serrano','675598001','916421756','C/azores','Fuenlabrada','Madrid','Madrid','Spain','28946',11,50000),
    ('Madrileña de riegos','Jose','Tacaño','655983045','916689215','C/Lagañas','Fuenlabrada','Madrid','Madrid','Spain','28943',11,20000),
    ('Lasas S.A.','Antonio','Lasas','34916540145','34914851312','C/Leganes 15',None,'Fuenlabrada','Madrid','Spain','28945',8,154310),
    ('Camunas Jardines S.L.','Pedro','Camunas','34914873241','34914871541','C/Virgenes 45','C/Princesas 2 1ºB','San Lorenzo del Escorial','Madrid','Spain','28145',8,16481),
    ('Dardena S.A.','Juan','Rodriguez','34912453217','34912484764','C/Nueva York 74',None,'Madrid','Madrid','Spain','28003',8,321000),
    ('Jardin de Flores','Javier','Villar','654865643','914538776','C/ Oña 34',None,'Madrid','Madrid','Spain','28950',30,40000),
    ('Flores Marivi','Maria','Rodriguez','666555444','912458657','C/Leganes24',None,'Fuenlabrada','Madrid','Spain','28945',5,1500),
    ('Flowers, S.A','Beatriz','Fernandez','698754159','978453216','C/Luis Salquillo4',None,'Montornes del valles','Barcelona','Spain','24586',5,3500),
    ('Naturajardin','Victoria','Cruz','612343529','916548735','Plaza Magallón 15',None,'Madrid','Madrid','Spain','28011',30,5050),
    ('Golf S.A.','Luis','Martinez','916458762','912354475','C/Estancado',None,'Santa cruz de Tenerife','Islas Canarias','Spain','38297',12,30000),
    ('Americh Golf Management SL','Mario','Suarez','964493072','964493063','C/Letardo',None,'Barcelona','Cataluña','Spain','12320',12,20000),
    ('Aloha','Cristian','Rodrigez','916485852','914489898','C/Roman 3',None,'Canarias','Canarias','Spain','35488',12,50000),
    ('El Prat','Francisco','Camacho','916882323','916493211','Avenida Tibidabo',None,'Barcelona','Cataluña','Spain','12320',12,30000),
    ('Sotogrande','Maria','Santillana','915576622','914825645','C/Paseo del Parque',None,'Sotogrande','Cadiz','Spain','11310',12,60000),
    ('Vivero Humanes','Federico','Gomez','654987690','916040875','C/Miguel Echegaray 54',None,'Humanes','Madrid','Spain','28970',30,7430),
    ('Fuenla City','Tony','Muñoz Mena','675842139','915483754','C/Callo 52',None,'Fuenlabrada','Madrid','Spain','28574',5,4500),
    ('Jardines y Mansiones Cactus SL','Eva María','Sánchez','916877445','914477777','Polígono Industrial Maspalomas, Nº52','Móstoles','Madrid','Madrid','Spain','29874',9,76000),
    ('Jardinerías Matías SL','Matías','San Martín','916544147','917897474','C/Francisco Arce, Nº44','Bustarviejo','Madrid','Madrid','Spain','37845',9,100500),
    ('Agrojardin','Benito','Lopez','675432926','916549264','C/Mar Caspio 43',None,'Getafe','Madrid','Spain','28904',30,8040),
    ('Top Campo','Joseluis','Sanchez','685746512','974315924','C/Ibiza 32',None,'Humanes','Madrid','Spain','28574',5,5500),
    ('Jardineria Sara','Sara','Marquez','675124537','912475843','C/Lima 1',None,'Fuenlabrada','Madrid','Spain','27584',5,7500),
    ('Campohermoso','Luis','Jimenez','645925376','916159116','C/Peru 78',None,'Fuenlabrada','Madrid','Spain','28945',30,3250),
    ('france telecom','François','Toulou','(33)5120578961','(33)5120578961','6 place d Alleray 15ème',None,'Paris',None,'France','75010',16,10000),
    ('Musée du Louvre','Pierre','Delacroux','(33)0140205050','(33)0140205442','Quai du Louvre',None,'Paris',None,'France','75058',16,30000),
    ('Tutifruti S.A','Jacob','Jones','2 9261-2433','2 9283-1695','level 24, St. Martins Tower.-31 Market St.',None,'Sydney','Nueva Gales del Sur','Australia','2000',31,10000),
    ('Flores S.L.','Antonio','Romero','654352981','685249700','Avenida España',None,'Madrid','Fuenlabrada','Spain','29643',18,6000),
    ('The Magic Garden','Richard','Mcain','926523468','9364875882','Lihgting Park',None,'London','London','United Kingdom','65930',18,10000),
    ('El Jardin Viviente S.L','Justin','Smith','2 8005-7161','2 8005-7162','176 Cumberland Street The rocks',None,'Sydney','Nueva Gales del Sur','Australia','2003',31,8000),
]
cur.executemany("INSERT INTO cliente(nombre_cliente,nombre_contacto,apellido_contacto,telefono,fax,linea_direccion1,linea_direccion2,ciudad,region,pais,codigo_postal,ID_empleado_rep_ventas,limite_credito) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)", clientes)

conn_j.commit()
print(f" Oficinas, Empleados, Categorías y Clientes insertados")

 Oficinas, Empleados, Categorías y Clientes insertados


## CELDA 4 — Poblar Pedidos, Productos, Detalle y Pagos

In [ ]:
# ─── PEDIDOS ─────────────────────────────────────────────────
pedidos = [
    ('2006-01-17','2006-01-19','2006-01-19','Entregado','Pagado a plazos',5),
    ('2007-10-23','2007-10-28','2007-10-26','Entregado','La entrega llego antes de lo esperado',5),
    ('2008-06-20','2008-06-25',None,'Rechazado','Limite de credito superado',5),
    ('2009-01-20','2009-01-26',None,'Pendiente',None,5),
    ('2008-11-09','2008-11-14','2008-11-14','Entregado','El cliente paga la mitad con tarjeta y la otra mitad con efectivo',1),
    ('2008-12-22','2008-12-27','2008-12-28','Entregado','El cliente comprueba la integridad del paquete, todo correcto',1),
    ('2009-01-15','2009-01-20',None,'Pendiente','El cliente llama para confirmar la fecha',3),
    ('2009-01-20','2009-01-27',None,'Pendiente','El cliente requiere entrega de 16:00h a 22:00h',1),
    ('2009-01-22','2009-01-27',None,'Pendiente','El cliente requiere entrega de 9:00h a 13:00h',1),
    ('2009-01-12','2009-01-14','2009-01-15','Entregado',None,7),
    ('2009-01-02','2009-01-02',None,'Rechazado','mal pago',7),
    ('2009-01-09','2009-01-12','2009-01-11','Entregado',None,7),
    ('2009-01-06','2009-01-07','2009-01-15','Entregado',None,7),
    ('2009-01-08','2009-01-09','2009-01-11','Entregado','mal estado',7),
    ('2009-01-05','2009-01-06','2009-01-07','Entregado',None,9),
    ('2009-01-18','2009-02-12',None,'Pendiente','entregar en murcia',9),
    ('2009-01-20','2009-02-15',None,'Pendiente',None,9),
    ('2009-01-09','2009-01-09','2009-01-09','Rechazado','mal pago',9),
    ('2009-01-11','2009-01-11','2009-01-13','Entregado',None,9),
    ('2008-12-30','2009-01-10',None,'Rechazado','El pedido fue anulado por el cliente',5),
    ('2008-07-14','2008-07-31','2008-07-25','Entregado',None,14),
    ('2009-02-02','2009-02-08',None,'Rechazado','El cliente carece de saldo en la cuenta asociada',1),
    ('2009-02-06','2009-02-12',None,'Rechazado','El cliente anula la operacion para adquirir mas producto',3),
    ('2009-02-07','2009-02-13',None,'Entregado','El pedido aparece como entregado pero no sabemos en que fecha',3),
    ('2009-01-10','2009-01-15','2009-01-18','Entregado',None,5),
    ('2009-02-06','2009-02-11','2009-02-12','Entregado',None,5),
    ('2009-01-13','2009-01-13','2009-01-17','Entregado',None,13),
    ('2009-01-13','2009-01-13','2009-01-13','Entregado',None,13),
    ('2009-01-15','2009-01-15','2009-01-15','Entregado',None,15),
    ('2009-01-16','2009-01-16','2009-01-26','Entregado',None,15),
    ('2009-02-16','2009-02-19',None,'Pendiente',None,16),
    ('2009-02-16','2009-02-19',None,'Pendiente',None,16),
    ('2009-02-09','2009-02-09',None,'Pendiente',None,27),
    ('2009-01-13','2009-01-15','2009-01-15','Entregado',None,28),
    ('2007-10-05','2007-10-15','2007-10-13','Entregado',None,35),
    ('2006-05-10','2006-05-20','2006-05-20','Entregado',None,36),
    ('2009-03-26','2009-03-26',None,'Pendiente',None,23),
    ('2009-01-16','2009-01-19','2009-01-19','Entregado',None,30),
    ('2009-03-06','2009-03-11',None,'Pendiente',None,19),
    ('2009-02-06','2009-02-11',None,'Pendiente',None,26),
    ('2009-02-08','2009-02-12','2009-02-12','Entregado',None,27),
    ('2009-01-06','2009-01-07',None,'Rechazado','Enviado con documentación de otro cliente',9),
    ('2009-01-06','2009-01-07',None,'Rechazado','Enviado con documentación de otro cliente',9),
    ('2009-01-06','2009-01-07',None,'Rechazado','Enviado con documentación de otro cliente',9),
    ('2009-01-05','2009-01-06',None,'Rechazado',None,9),
    ('2009-01-05','2009-01-06',None,'Rechazado',None,9),
    ('2009-01-05','2009-01-06',None,'Rechazado',None,9),
    ('2009-01-05','2009-01-06',None,'Rechazado',None,9),
    ('2009-01-05','2009-01-06',None,'Rechazado',None,9),
    ('2009-01-05','2009-01-06',None,'Rechazado',None,9),
    ('2009-01-05','2009-01-06',None,'Rechazado',None,9),
    ('2009-01-05','2009-01-06',None,'Rechazado',None,9),
    ('2009-01-05','2009-01-06',None,'Rechazado',None,9),
    ('2009-01-05','2009-01-06',None,'Rechazado',None,9),
    ('2009-01-05','2009-01-06',None,'Rechazado',None,9),
    ('2009-01-05','2009-01-06',None,'Rechazado',None,9),
    ('2009-01-05','2009-01-06',None,'Rechazado',None,9),
    ('2009-01-05','2009-01-06',None,'Rechazado',None,9),
    ('2009-01-05','2009-01-06',None,'Rechazado',None,9),
    ('2009-01-05','2009-01-06',None,'Rechazado',None,9),
    ('2009-01-05','2009-01-06',None,'Rechazado',None,9),
    ('2009-01-05','2009-01-06',None,'Rechazado',None,9),
    ('2009-01-05','2009-01-06',None,'Rechazado',None,9),
    ('2009-01-05','2009-01-06',None,'Rechazado',None,9),
    ('2009-01-05','2009-01-06',None,'Rechazado',None,9),
    ('2009-01-05','2009-01-06',None,'Rechazado',None,9),
    ('2009-01-05','2009-01-06',None,'Rechazado',None,9),
    ('2009-01-09','2009-01-09',None,'Rechazado','mal pago',9),
    ('2009-01-09','2009-01-09',None,'Rechazado','mal pago',9),
    ('2009-01-09','2009-01-09',None,'Rechazado','mal pago',9),
    ('2009-01-09','2009-01-09',None,'Rechazado','mal pago',9),
    ('2009-01-09','2009-01-09',None,'Rechazado','mal pago',9),
    ('2009-01-09','2009-01-09',None,'Rechazado','mal pago',9),
    ('2009-01-09','2009-01-09',None,'Rechazado','mal pago',9),
    ('2009-01-09','2009-01-09',None,'Rechazado','mal pago',9),
    ('2009-01-09','2009-01-09',None,'Rechazado','mal pago',9),
    ('2009-01-09','2009-01-09',None,'Rechazado','mal pago',9),
    ('2009-01-09','2009-01-09',None,'Rechazado','mal pago',9),
    ('2009-01-09','2009-01-09',None,'Rechazado','mal pago',9),
    ('2009-01-09','2009-01-09',None,'Rechazado','mal pago',9),
    ('2009-01-09','2009-01-09',None,'Rechazado','mal pago',9),
    ('2009-01-09','2009-01-09',None,'Rechazado','mal pago',9),
    ('2009-01-09','2009-01-09',None,'Rechazado','mal pago',9),
    ('2009-01-09','2009-01-09',None,'Rechazado','mal pago',9),
    ('2009-01-09','2009-01-09',None,'Rechazado','mal pago',9),
    ('2009-01-09','2009-01-09',None,'Rechazado','mal pago',9),
    ('2009-01-09','2009-01-09',None,'Rechazado','mal pago',9),
    ('2009-01-09','2009-01-09',None,'Rechazado','mal pago',9),
    ('2009-01-09','2009-01-09',None,'Rechazado','mal pago',9),
    ('2009-01-09','2009-01-09',None,'Rechazado','mal pago',9),
    ('2009-01-09','2009-01-09',None,'Rechazado','mal pago',9),
    ('2009-01-09','2009-01-09',None,'Rechazado','mal pago',9),
    ('2009-01-09','2009-01-09',None,'Rechazado','mal pago',9),
]
cur.executemany("INSERT INTO pedido(fecha_pedido,fecha_esperada,fecha_entrega,estado,comentarios,ID_cliente) VALUES (?,?,?,?,?,?)", pedidos)

# ─── PRODUCTOS (muestra representativa de todas las categorías) ─
productos = [
    ('AR-001','Ajedrea','3','',"Murcia Seasons",'',140,1,0.55),
    ('AR-002','Albahaca','3','',"Murcia Seasons",'',840,1,0.55),
    ('AR-003','Alcaravea','3','',"Murcia Seasons",'',140,1,0.55),
    ('AR-004','Anís','3','',"Murcia Seasons",'',140,1,0.55),
    ('AR-005','Borraja','3','',"Murcia Seasons",'',140,1,0.55),
    ('HE-001','Gazania','1','40-50cm','Viveros Frabosa','Margaritas en colores brillantes',100,5,3.5),
    ('HE-002','Lantana camara','1','45-50cm','Viveros Frabosa','Flores en racimos de colores',120,8,5),
    ('HER-001','Azada de acero forjado','2','','Herramientas Premium','Azada profesional',50,22,18),
    ('HER-002','Pala de plantación','2','','Herramientas Premium','Pala con mango ergonómico',60,19,15),
    ('FR-1','Cerezo','4','8/10','Frutales Talavera S.A','Prunus avium',50,16,13),
    ('FR-2','Cerezo','4','10/12','Frutales Talavera S.A','Prunus avium',50,27,22),
    ('FR-3','Cerezo','4','12/14','Frutales Talavera S.A','Prunus avium',50,38,30),
    ('FR-4','Cerezo','4','14/16','Frutales Talavera S.A','Prunus avium',50,49,39),
    ('FR-5','Almendro','4','8/10','Frutales Talavera S.A','',50,11,8),
    ('FR-6','Almendro','4','10/12','Frutales Talavera S.A','',50,22,17),
    ('FR-7','Almendro','4','12/14','Frutales Talavera S.A','',50,32,25),
    ('FR-8','Almendro','4','14/16','Frutales Talavera S.A','',50,49,39),
    ('FR-9','Limonero calibre 8/10','4','','NaranjasValencianas.com','',15,29,23),
    ('FR-10','Limonero calibre 10/12','4','','NaranjasValencianas.com','',15,38,30),
    ('FR-11','Limonero calibre 12/14','4','','NaranjasValencianas.com','',15,49,39),
    ('OR-001','Arbustos Mix Maceta','5','40-60','Valencia Garden Service','',25,5,4),
    ('OR-100','Mimosa Injerto CLASICA Dealbata','5','100-110','Viveros EL OASIS','Acacia dealbata',100,12,9),
    ('OR-108','Abelia Floribunda','5','35-45','Viveros EL OASIS','',100,5,4),
    ('OR-109','Callistemom (Mix)','5','35-45','Viveros EL OASIS','Limpitatubos',100,5,4),
    ('OR-116','Hibiscus Syriacus Diana','5','35-45','Viveros EL OASIS','Flores de muchos colores',120,7,5),
    ('OR-127','Camelia japonica','5','40-60','Viveros EL OASIS','Arbusto excepcional',50,7,5),
    ('OR-133','Nerium oleander CALIDAD GARDEN','5','40-45','Viveros EL OASIS','',50,2,1),
    ('OR-137','ROSAL TREPADOR','5','','Viveros EL OASIS','',100,4,3),
    ('OR-143','Wisteria Sinensis azul, rosa, blanca','5','','Viveros EL OASIS','',100,9,7),
    ('OR-145','Bougamvillea Sanderiana Tutor','5','80-100','Viveros EL OASIS','',100,2,1),
]
cur.executemany("INSERT INTO producto(CodigoProducto,nombre,Categoria,dimensiones,proveedor,descripcion,cantidad_en_stock,precio_venta,precio_proveedor) VALUES (?,?,?,?,?,?,?,?,?)", productos)

# ─── DETALLE PEDIDOS ─────────────────────────────────────────
detalles = [
    (1,1,4,12,1), (1,2,16,12,2), (2,3,20,12,1), (2,4,35,12,2),
    (3,5,30,9,1), (3,6,10,9,2), (4,7,50,9,1), (4,8,15,9,2),
    (5,9,30,5,1), (5,10,25,29,2), (6,11,40,4,1), (6,12,5,4,2),
    (7,13,20,38,1), (7,14,30,49,2), (8,15,10,16,1), (8,16,18,27,2),
    (9,17,35,32,1), (9,18,20,38,2), (10,19,40,49,1), (10,20,25,22,2),
    (11,21,10,5,1), (12,22,20,7,1), (13,23,15,19,1), (14,24,30,5,1),
    (15,25,50,7,1), (15,26,20,7,2), (16,27,40,9,1), (16,28,10,4,2),
    (17,29,25,2,1), (17,30,35,9,2), (18,1,15,12,1), (18,2,20,12,2),
    (19,3,10,12,1), (20,4,25,12,1), (21,5,30,9,1), (21,6,15,9,2),
    (22,7,20,9,1), (22,8,40,9,2), (23,9,10,5,1), (23,10,30,29,2),
    (24,11,25,4,1), (25,12,35,4,1), (26,13,20,38,1), (26,14,40,49,2),
    (27,15,15,16,1), (28,16,25,27,1), (29,17,30,32,1), (29,18,20,38,2),
    (30,19,10,49,1), (30,20,35,22,2),
]
cur.executemany("INSERT INTO detalle_pedido(ID_pedido,ID_producto,cantidad,precio_unidad,numero_linea) VALUES (?,?,?,?,?)", detalles)

# ─── PAGOS ───────────────────────────────────────────────────
pagos = [
    (1,'PayPal','ak-std-000001','2008-11-10',2000),
    (1,'PayPal','ak-std-000002','2008-12-10',2000),
    (3,'PayPal','ak-std-000003','2009-01-16',5000),
    (3,'PayPal','ak-std-000004','2009-02-16',5000),
    (3,'PayPal','ak-std-000005','2009-02-19',926),
    (4,'PayPal','ak-std-000006','2007-01-08',20000),
    (4,'PayPal','ak-std-000007','2007-01-08',20000),
    (4,'PayPal','ak-std-000008','2007-01-08',20000),
    (4,'PayPal','ak-std-000009','2007-01-08',20000),
    (4,'PayPal','ak-std-000010','2007-01-08',1849),
    (5,'Transferencia','ak-std-000011','2006-01-18',23794),
    (7,'Cheque','ak-std-000012','2009-01-13',2390),
    (9,'PayPal','ak-std-000013','2009-01-06',929),
    (13,'PayPal','ak-std-000014','2008-08-04',2246),
    (14,'PayPal','ak-std-000015','2008-07-15',4160),
    (15,'PayPal','ak-std-000016','2009-01-15',2081),
    (15,'PayPal','ak-std-000035','2009-02-15',10000),
    (16,'PayPal','ak-std-000017','2009-02-16',4399),
    (19,'PayPal','ak-std-000018','2009-03-06',232),
    (23,'PayPal','ak-std-000019','2009-03-26',272),
    (26,'PayPal','ak-std-000020','2008-03-18',18846),
    (27,'PayPal','ak-std-000021','2009-02-08',10972),
    (28,'PayPal','ak-std-000022','2009-01-13',8489),
    (30,'PayPal','ak-std-000024','2009-01-16',7863),
    (35,'PayPal','ak-std-000025','2007-10-06',3321),
    (36,'PayPal','ak-std-000026','2006-05-26',1171),
]
cur.executemany("INSERT INTO pago(ID_cliente,forma_pago,id_transaccion,fecha_pago,total) VALUES (?,?,?,?,?)", pagos)

conn_j.commit()
print(" Pedidos, Productos, Detalles y Pagos insertados")

 Pedidos, Productos, Detalles y Pagos insertados


## CELDA 5 — Verificar Jardinería

In [ ]:
conteos_j = """
SELECT 'oficina'          AS tabla, COUNT(*) AS total FROM oficina
UNION ALL SELECT 'empleado',       COUNT(*) FROM empleado
UNION ALL SELECT 'Categoria_prod', COUNT(*) FROM Categoria_producto
UNION ALL SELECT 'cliente',        COUNT(*) FROM cliente
UNION ALL SELECT 'pedido',         COUNT(*) FROM pedido
UNION ALL SELECT 'producto',       COUNT(*) FROM producto
UNION ALL SELECT 'detalle_pedido', COUNT(*) FROM detalle_pedido
UNION ALL SELECT 'pago',           COUNT(*) FROM pago
"""
run_query(conn_j, conteos_j, " Conteo de registros — Jardinería")


 Conteo de registros — Jardinería


,tabla,total
0,oficina,9
1,empleado,31
2,Categoria_prod,5
3,cliente,36
4,pedido,93
5,producto,30
6,detalle_pedido,50
7,pago,26


,tabla,total
0,oficina,9
1,empleado,31
2,Categoria_prod,5
3,cliente,36
4,pedido,93
5,producto,30
6,detalle_pedido,50
7,pago,26


## CELDA 6 — Crear base de datos Staging

In [ ]:
if os.path.exists(DB_STAGING):
    os.remove(DB_STAGING)
    print("  staging.db eliminada (reinicio limpio)")

conn_s = sqlite3.connect(DB_STAGING)
conn_s.execute("PRAGMA foreign_keys = ON")

ddl_staging = """
-- ===========================================
-- BASE DE DATOS STAGING - STAR SCHEMA
-- ===========================================

-- Dimensión Oficinas
CREATE TABLE dim_oficina (
    ID_oficina    INTEGER PRIMARY KEY,
    descripcion   VARCHAR(10),
    ciudad        VARCHAR(30),
    pais          VARCHAR(50),
    region        VARCHAR(50),
    codigo_postal VARCHAR(10),
    telefono      VARCHAR(20)
);

-- Dimensión Empleados
CREATE TABLE dim_empleado (
    ID_empleado     INTEGER PRIMARY KEY,
    nombre_completo VARCHAR(150),
    email           VARCHAR(100),
    puesto          VARCHAR(50),
    ID_oficina      INTEGER,
    ID_jefe         INTEGER
);

-- Dimensión Clientes
CREATE TABLE dim_cliente (
    ID_cliente      INTEGER PRIMARY KEY,
    nombre_cliente  VARCHAR(50),
    nombre_contacto VARCHAR(60),
    telefono        VARCHAR(15),
    ciudad          VARCHAR(50),
    pais            VARCHAR(50),
    limite_credito  NUMERIC(15,2),
    ID_empleado_rep INTEGER
);

-- Dimensión Categorías
CREATE TABLE dim_categoria (
    ID_categoria INTEGER PRIMARY KEY,
    descripcion  VARCHAR(50)
);

-- Dimensión Productos
CREATE TABLE dim_producto (
    ID_producto       INTEGER PRIMARY KEY,
    codigo            VARCHAR(15),
    nombre            VARCHAR(70),
    ID_categoria      INTEGER,
    proveedor         VARCHAR(50),
    cantidad_en_stock INTEGER,
    precio_venta      NUMERIC(15,2),
    precio_proveedor  NUMERIC(15,2)
);

-- Tabla de hechos: Pedidos
CREATE TABLE fact_pedidos (
    ID_detalle    INTEGER PRIMARY KEY,
    ID_pedido     INTEGER,
    fecha_pedido  DATE,
    fecha_entrega DATE,
    estado        VARCHAR(15),
    ID_cliente    INTEGER,
    ID_producto   INTEGER,
    cantidad      INTEGER,
    precio_unidad NUMERIC(15,2),
    total_linea   NUMERIC(15,2),
    numero_linea  INTEGER
);

-- Tabla de hechos: Pagos
CREATE TABLE fact_pagos (
    ID_pago        INTEGER PRIMARY KEY,
    ID_cliente     INTEGER,
    forma_pago     VARCHAR(40),
    id_transaccion VARCHAR(50),
    fecha_pago     DATE,
    total          NUMERIC(15,2)
);
"""

ejecutar_ddl(conn_s, ddl_staging, "Tablas de Staging creadas")

  staging.db eliminada (reinicio limpio)
✅ Tablas de Staging creadas


## CELDA 7 — Migrar datos de Jardinería → Staging

In [ ]:
# Conectar las dos bases en la misma sesión con ATTACH
conn_s.execute(f"ATTACH DATABASE '{DB_JARDINERIA}' AS jard")

migraciones = [
    (
        "dim_oficina",
        """INSERT INTO dim_oficina
           SELECT ID_oficina, Descripcion, ciudad, pais, region, codigo_postal, telefono
           FROM jard.oficina"""
    ),
    (
        "dim_empleado",
        """INSERT INTO dim_empleado
           SELECT
             ID_empleado,
             TRIM(nombre || ' ' || apellido1 ||
               CASE WHEN COALESCE(apellido2,'') <> '' THEN ' ' || apellido2 ELSE '' END),
             email, puesto, ID_oficina, ID_jefe
           FROM jard.empleado"""
    ),
    (
        "dim_cliente",
        """INSERT INTO dim_cliente
           SELECT
             ID_cliente, nombre_cliente,
             TRIM(COALESCE(nombre_contacto,'') || ' ' || COALESCE(apellido_contacto,'')),
             telefono, ciudad, pais, limite_credito, ID_empleado_rep_ventas
           FROM jard.cliente"""
    ),
    (
        "dim_categoria",
        """INSERT INTO dim_categoria
           SELECT Id_Categoria, Desc_Categoria
           FROM jard.Categoria_producto"""
    ),
    (
        "dim_producto",
        """INSERT INTO dim_producto
           SELECT ID_producto, CodigoProducto, nombre, Categoria,
                  proveedor, cantidad_en_stock, precio_venta, precio_proveedor
           FROM jard.producto"""
    ),
    (
        "fact_pedidos",
        """INSERT INTO fact_pedidos
           SELECT
             dp.ID_detalle_pedido,
             dp.ID_pedido,
             p.fecha_pedido,
             p.fecha_entrega,
             p.estado,
             p.ID_cliente,
             dp.ID_producto,
             dp.cantidad,
             dp.precio_unidad,
             ROUND(dp.cantidad * dp.precio_unidad, 2),
             dp.numero_linea
           FROM jard.detalle_pedido dp
           JOIN jard.pedido p ON dp.ID_pedido = p.ID_pedido"""
    ),
    (
        "fact_pagos",
        """INSERT INTO fact_pagos
           SELECT ID_pago, ID_cliente, forma_pago, id_transaccion, fecha_pago, total
           FROM jard.pago"""
    ),
]

for tabla, sql in migraciones:
    try:
        conn_s.execute(sql)
        conn_s.commit()
        cnt = conn_s.execute(f"SELECT COUNT(*) FROM {tabla}").fetchone()[0]
        print(f" {tabla:<20} → {cnt} filas migradas")
    except Exception as e:
        print(f" Error en {tabla}: {e}")

 dim_oficina          → 9 filas migradas
 dim_empleado         → 31 filas migradas
 dim_cliente          → 36 filas migradas
 dim_categoria        → 5 filas migradas
 dim_producto         → 30 filas migradas
 fact_pedidos         → 50 filas migradas
 fact_pagos           → 26 filas migradas


## CELDA 8 — Validaciones completas

In [ ]:
print("\n" + "="*55)
print("VALIDACIÓN 1: Conteo de filas por tabla — Staging")
print("="*55)
conteos_s = """
SELECT 'dim_oficina'   AS tabla, COUNT(*) AS filas FROM dim_oficina
UNION ALL SELECT 'dim_empleado',  COUNT(*) FROM dim_empleado
UNION ALL SELECT 'dim_cliente',   COUNT(*) FROM dim_cliente
UNION ALL SELECT 'dim_categoria', COUNT(*) FROM dim_categoria
UNION ALL SELECT 'dim_producto',  COUNT(*) FROM dim_producto
UNION ALL SELECT 'fact_pedidos',  COUNT(*) FROM fact_pedidos
UNION ALL SELECT 'fact_pagos',    COUNT(*) FROM fact_pagos
"""
run_query(conn_s, conteos_s)

print("\n" + "="*55)
print("VALIDACIÓN 2: Ventas totales por categoría de producto")
print("="*55)
run_query(conn_s, """
SELECT
    c.descripcion AS categoria,
    COUNT(DISTINCT f.ID_pedido)    AS num_pedidos,
    SUM(f.cantidad)                AS unidades_vendidas,
    ROUND(SUM(f.total_linea), 2)   AS ingresos_totales
FROM fact_pedidos f
JOIN dim_producto p ON f.ID_producto = p.ID_producto
JOIN dim_categoria c ON p.ID_categoria = c.ID_categoria
GROUP BY c.descripcion
ORDER BY ingresos_totales DESC
""")

print("\n" + "="*55)
print("VALIDACIÓN 3: Top 5 clientes por volumen de pagos")
print("="*55)
run_query(conn_s, """
SELECT
    cl.nombre_cliente,
    cl.pais,
    COUNT(fp.ID_pago)           AS num_pagos,
    ROUND(SUM(fp.total), 2)     AS total_pagado
FROM fact_pagos fp
JOIN dim_cliente cl ON fp.ID_cliente = cl.ID_cliente
GROUP BY cl.ID_cliente
ORDER BY total_pagado DESC
LIMIT 5
""")

print("\n" + "="*55)
print("VALIDACIÓN 4: Pedidos por estado")
print("="*55)
run_query(conn_s, """
SELECT estado, COUNT(*) AS cantidad
FROM fact_pedidos
GROUP BY estado
ORDER BY cantidad DESC
""")

print("\n" + "="*55)
print("VALIDACIÓN 5: Formas de pago utilizadas")
print("="*55)
run_query(conn_s, """
SELECT forma_pago, COUNT(*) AS veces_usado, ROUND(SUM(total),2) AS total
FROM fact_pagos
GROUP BY forma_pago
ORDER BY total DESC
""")


VALIDACIÓN 1: Conteo de filas por tabla — Staging


,tabla,filas
0,dim_oficina,9
1,dim_empleado,31
2,dim_cliente,36
3,dim_categoria,5
4,dim_producto,30
5,fact_pedidos,50
6,fact_pagos,26



VALIDACIÓN 2: Ventas totales por categoría de producto


,categoria,num_pedidos,unidades_vendidas,ingresos_totales
0,Frutales,14,553,15896.0
1,Aromaticas,7,205,2280.0
2,Ornamentales,7,255,1880.0
3,Herbaceas,4,95,855.0
4,Herramientas,4,95,695.0



VALIDACIÓN 3: Top 5 clientes por volumen de pagos


...

,estado,cantidad
0,Entregado,28
1,Pendiente,12
2,Rechazado,10



VALIDACIÓN 5: Formas de pago utilizadas


,forma_pago,veces_usado,total
0,PayPal,24,171756.0
1,Transferencia,1,23794.0
2,Cheque,1,2390.0


,forma_pago,veces_usado,total
0,PayPal,24,171756.0
1,Transferencia,1,23794.0
2,Cheque,1,2390.0


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


## CELDA 9 — Exportar BK y script SQL para descargar

In [ ]:
# ─── 1. Cerrar conexiones para que los .db queden escritos ───
conn_j.close()
conn_s.close()

# ─── 2. Copiar los .db como BK ───────────────────────────────
shutil.copy(DB_JARDINERIA, '/content/BK_jardineria.db')
shutil.copy(DB_STAGING,    '/content/BK_staging.db')
print(" BK_jardineria.db generado")
print(" BK_staging.db generado")

# ─── 3. Generar el script SQL de Staging ─────────────────────
script_sql = '''
-- ============================================================
-- SCRIPT: Creación y carga de la base de datos STAGING
-- Motor:  SQLite (compatible con adaptación a SQL Server)
-- Autor:  Proyecto Jardinería
-- ============================================================

-- PARTE 1: ESTRUCTURA (DDL)
-- ============================================================

CREATE TABLE IF NOT EXISTS dim_oficina (
    ID_oficina    INTEGER PRIMARY KEY,
    descripcion   VARCHAR(10),
    ciudad        VARCHAR(30),
    pais          VARCHAR(50),
    region        VARCHAR(50),
    codigo_postal VARCHAR(10),
    telefono      VARCHAR(20)
);

CREATE TABLE IF NOT EXISTS dim_empleado (
    ID_empleado     INTEGER PRIMARY KEY,
    nombre_completo VARCHAR(150),
    email           VARCHAR(100),
    puesto          VARCHAR(50),
    ID_oficina      INTEGER,
    ID_jefe         INTEGER
);

CREATE TABLE IF NOT EXISTS dim_cliente (
    ID_cliente      INTEGER PRIMARY KEY,
    nombre_cliente  VARCHAR(50),
    nombre_contacto VARCHAR(60),
    telefono        VARCHAR(15),
    ciudad          VARCHAR(50),
    pais            VARCHAR(50),
    limite_credito  NUMERIC(15,2),
    ID_empleado_rep INTEGER
);

CREATE TABLE IF NOT EXISTS dim_categoria (
    ID_categoria INTEGER PRIMARY KEY,
    descripcion  VARCHAR(50)
);

CREATE TABLE IF NOT EXISTS dim_producto (
    ID_producto       INTEGER PRIMARY KEY,
    codigo            VARCHAR(15),
    nombre            VARCHAR(70),
    ID_categoria      INTEGER,
    proveedor         VARCHAR(50),
    cantidad_en_stock INTEGER,
    precio_venta      NUMERIC(15,2),
    precio_proveedor  NUMERIC(15,2)
);

CREATE TABLE IF NOT EXISTS fact_pedidos (
    ID_detalle    INTEGER PRIMARY KEY,
    ID_pedido     INTEGER,
    fecha_pedido  DATE,
    fecha_entrega DATE,
    estado        VARCHAR(15),
    ID_cliente    INTEGER,
    ID_producto   INTEGER,
    cantidad      INTEGER,
    precio_unidad NUMERIC(15,2),
    total_linea   NUMERIC(15,2),
    numero_linea  INTEGER
);

CREATE TABLE IF NOT EXISTS fact_pagos (
    ID_pago        INTEGER PRIMARY KEY,
    ID_cliente     INTEGER,
    forma_pago     VARCHAR(40),
    id_transaccion VARCHAR(50),
    fecha_pago     DATE,
    total          NUMERIC(15,2)
);

-- ============================================================
-- PARTE 2: MIGRACIÓN DE DATOS (DML)
-- Asume que jardineria está disponible como base adjunta
-- En SQLite: ATTACH DATABASE 'jardineria.db' AS jard;
-- ============================================================

INSERT INTO dim_oficina
SELECT ID_oficina, Descripcion, ciudad, pais, region, codigo_postal, telefono
FROM jard.oficina;

INSERT INTO dim_empleado
SELECT
  ID_empleado,
  TRIM(nombre || \' \' || apellido1 ||
    CASE WHEN COALESCE(apellido2,\'\') <> \'\' THEN \' \' || apellido2 ELSE \'\' END),
  email, puesto, ID_oficina, ID_jefe
FROM jard.empleado;

INSERT INTO dim_cliente
SELECT
  ID_cliente, nombre_cliente,
  TRIM(COALESCE(nombre_contacto,\'\') || \' \' || COALESCE(apellido_contacto,\'\'))
  , telefono, ciudad, pais, limite_credito, ID_empleado_rep_ventas
FROM jard.cliente;

INSERT INTO dim_categoria
SELECT Id_Categoria, Desc_Categoria FROM jard.Categoria_producto;

INSERT INTO dim_producto
SELECT ID_producto, CodigoProducto, nombre, Categoria,
       proveedor, cantidad_en_stock, precio_venta, precio_proveedor
FROM jard.producto;

INSERT INTO fact_pedidos
SELECT
  dp.ID_detalle_pedido, dp.ID_pedido,
  p.fecha_pedido, p.fecha_entrega, p.estado, p.ID_cliente,
  dp.ID_producto, dp.cantidad, dp.precio_unidad,
  ROUND(dp.cantidad * dp.precio_unidad, 2),
  dp.numero_linea
FROM jard.detalle_pedido dp
JOIN jard.pedido p ON dp.ID_pedido = p.ID_pedido;

INSERT INTO fact_pagos
SELECT ID_pago, ID_cliente, forma_pago, id_transaccion, fecha_pago, total
FROM jard.pago;
'''

with open('/content/Anexo_Script_Staging.sql', 'w', encoding='utf-8') as f:
    f.write(script_sql)
print(" Anexo_Script_Staging.sql generado")

print("\n Archivos listos para descargar:")
for archivo in ['BK_jardineria.db','BK_staging.db','Anexo_Script_Staging.sql']:
    size = os.path.getsize(f'/content/{archivo}')
    print(f"   {archivo}  ({size:,} bytes)")

 BK_jardineria.db generado
 BK_staging.db generado
 Anexo_Script_Staging.sql generado

 Archivos listos para descargar:
   BK_jardineria.db  (49,152 bytes)
   BK_staging.db  (32,768 bytes)
   Anexo_Script_Staging.sql  (3,786 bytes)


## CELDA 10 — Descargar todos los archivos

In [36]:
print("⬇  Descargando archivos...")
files.download('/content/BK_jardineria.db')
files.download('/content/BK_staging.db')
files.download('/content/Anexo_Script_Staging.sql')
print(" Listo. Revisa la carpeta de Descargas de tu computador.")

⬇  Descargando archivos...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 Listo. Revisa la carpeta de Descargas de tu computador.


---
##  Resumen del análisis de migración

| Tabla Jardinería | Tabla Staging | Tipo | Campos excluidos | Razón |
|---|---|---|---|---|
| `oficina` | `dim_oficina` | Dimensión | `linea_direccion1/2` | No relevantes para análisis BI |
| `empleado` | `dim_empleado` | Dimensión | `extension` | Dato operativo interno |
| `cliente` | `dim_cliente` | Dimensión | `fax`, `linea_direccion*`, `codigo_postal` | Datos de contacto, no analíticos |
| `Categoria_producto` | `dim_categoria` | Dimensión | `descripcion_texto/html`, `imagen` | Contenido web |
| `producto` | `dim_producto` | Dimensión | `dimensiones`, `descripcion` | Texto libre no estructurado |
| `pedido` + `detalle_pedido` | `fact_pedidos` | Hecho | `comentarios` | JOIN → tabla de hechos unificada |
| `pago` | `fact_pagos` | Hecho | — | Todos los campos son relevantes |

**Campo calculado agregado en Staging:**
- `total_linea = cantidad × precio_unidad` — columna derivada que facilita análisis sin recalcular.